# YOLO11s — Nhận diện 6 biển báo
Dataset: [trafficsign / Sa Tr](https://universe.roboflow.com/sa-tr/trafficsign-jryf1), version 1, CC BY 4.0.

**6 lớp:** `left`, `no_left`, `right`, `no_right`, `straight`, `stop`.

Chạy lần lượt từ trên xuống. Trên Colab: **Runtime → Change runtime type → GPU**. Cấu hình chung nằm ở bước 2; API key nằm trong `.env`.

## 1. Cài thư viện
**Windows local:** chọn kernel `.venv/Scripts/python.exe` của dự án. Môi trường này dùng NumPy wheel thay cho bản MKL đang xung đột với PyTorch trong `AI`.

Sau khi cài hoặc đổi thư viện, restart kernel **một lần** rồi chạy tiếp từ bước 2. Không chạy lại ô cài đặt trong khi đang train.

In [ ]:
%pip install -q "numpy==2.2.6" ultralytics roboflow python-dotenv pyyaml matplotlib pillow onnx onnxruntime

## 2. Kiểm tra thiết bị
Có GPU NVIDIA và PyTorch hỗ trợ CUDA thì dùng GPU số 0; nếu không, dùng CPU. Train trên CPU sẽ chậm. Trên Colab, hãy bật GPU trước khi train.

In [ ]:
from pathlib import Path
import os
import random
import shutil
import torch
import yaml
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from IPython.display import display
from ultralytics import YOLO
from roboflow import Roboflow
from dotenv import dotenv_values
import ultralytics
import importlib.metadata

# Cấu hình chung — chỉnh tại đây.
DATASET_WORKSPACE = "sa-tr"
DATASET_PROJECT = "trafficsign-jryf1"
DATASET_VERSION = 1
MODEL_NAME = "yolo11s.pt"
EPOCHS = 100
IMAGE_SIZE = 640
PATIENCE = 20
DEVICE = 0 if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8 if DEVICE == 0 else 2
print("Thiết bị:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU")
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("Roboflow:", importlib.metadata.version("roboflow"))

# Mọi dữ liệu và kết quả nằm trong thư mục này.
# Local: tìm thư mục gốc. Colab độc lập: dùng /content.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), Path.cwd().parent)
     if (path / "simulator").is_dir() and (path / "notebooks").is_dir()),
    Path.cwd(),
).resolve()
DATA_DIR = PROJECT_ROOT / "data" / "datasets"
WORK_DIR = PROJECT_ROOT / "outputs"
PRETRAINED_DIR = PROJECT_ROOT / "models" / "pretrained"
RUNS_DIR = WORK_DIR / "runs"
for folder in (DATA_DIR, WORK_DIR, PRETRAINED_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Thư mục dự án:", PROJECT_ROOT)

## 3. API key và tải dữ liệu
1. Đăng nhập [Roboflow API Settings](https://app.roboflow.com/settings/api), chọn workspace của bạn.
2. Copy **Private API Key** (hoặc **Generate New Key**).
3. Điền vào `.env`: `ROBOFLOW_API_KEY=key_cua_ban`.

**Local:** dùng `.env` ở thư mục gốc dự án. **Colab:** tải `.env` lên mục Files tại `/content`. Notebook không in key và `.env` đã được loại khỏi Git.

[Hướng dẫn chính thức](https://docs.roboflow.com/developer/authentication/find-your-roboflow-api-key).

In [ ]:
DATASET_DIR = DATA_DIR / f"{DATASET_PROJECT}-v{DATASET_VERSION}"
if (DATASET_DIR / "data.yaml").is_file():
    print("Dùng dataset đã tải:", DATASET_DIR)
else:
    env_paths = [PROJECT_ROOT / ".env", Path.cwd() / ".env"]
    env_file = next((path for path in env_paths if path.is_file()), None)
    credentials = dotenv_values(env_file) if env_file else {}
    api_key = (os.getenv("ROBOFLOW_API_KEY") or credentials.get("ROBOFLOW_API_KEY") or "").strip()
    if not api_key:
        raise ValueError("Điền ROBOFLOW_API_KEY trong .env rồi chạy lại ô này.")
    rf = Roboflow(api_key=api_key)
    project = rf.workspace(DATASET_WORKSPACE).project(DATASET_PROJECT)
    dataset = project.version(DATASET_VERSION).download("yolov11", location=str(DATASET_DIR))
    DATASET_DIR = Path(dataset.location).resolve()
    print("Dataset đã tải về:", DATASET_DIR)


## 4. Chuẩn bị train / validation / test
Đọc đường dẫn thực tế và tạo `data_local.yaml`. Nếu thiếu validation, chia ảnh train theo nhóm tên nguồn 80/10/10, seed 42; không di chuyển ảnh gốc. Các bản tăng cường `.rf.` cùng nguồn được giữ chung một tập. Cách nhóm này không bảo đảm tách được các frame gần nhau nếu chúng có tên nguồn khác nhau.

In [ ]:
ORIGINAL_YAML = DATASET_DIR / "data.yaml"
with ORIGINAL_YAML.open(encoding="utf-8") as f:
    original_config = yaml.safe_load(f)

names = original_config["names"]
CLASS_NAMES = dict(enumerate(names)) if isinstance(names, list) else {int(i): name for i, name in names.items()}
assert sorted(CLASS_NAMES) == list(range(len(CLASS_NAMES))), "ID lớp phải liên tục từ 0."

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
def get_images(source):
    if source.is_file() and source.suffix == ".txt":
        return [Path(line) for line in source.read_text(encoding="utf-8").splitlines() if line.strip()]
    return sorted(p for p in source.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)

def resolve_split(key, aliases):
    # Ưu tiên cấu hình YAML; dự phòng cấu trúc xuất thường gặp của Roboflow.
    raw = original_config.get(key)
    candidates = []
    if isinstance(raw, str) and raw:
        configured_root = Path(original_config.get("path") or ".")
        if not configured_root.is_absolute():
            configured_root = DATASET_DIR / configured_root
        candidates.append((configured_root / raw).resolve())
    candidates.extend(DATASET_DIR / alias / "images" for alias in aliases)
    return next((path for path in candidates if path.exists() and get_images(path)), None)

TRAIN_SOURCE = resolve_split("train", ["train"])
VAL_SOURCE = resolve_split("val", ["valid", "val"])
TEST_SOURCE = resolve_split("test", ["test"])
if TRAIN_SOURCE is None:
    raise FileNotFoundError(f"Không tìm thấy ảnh train trong {DATASET_DIR}")

if VAL_SOURCE is None:
    # Không sửa/di chuyển ảnh gốc; YOLO đọc danh sách ảnh trong các file TXT.
    # Các bản *.rf.<hash>.jpg cùng nguồn được giữ chung một tập.
    groups = {}
    for image_path in get_images(TRAIN_SOURCE):
        source_id = image_path.stem.split(".rf.")[0]
        groups.setdefault(source_id, []).append(image_path.resolve())
    group_ids = sorted(groups)
    random.Random(42).shuffle(group_ids)
    if len(group_ids) < 10:
        raise ValueError("Quá ít nhóm ảnh để tự chia tập; cần bổ sung hoặc chia dữ liệu thủ công.")
    train_end = int(len(group_ids) * 0.8)
    val_end = int(len(group_ids) * 0.9)
    partitions = {
        "train": group_ids[:train_end],
        "val": group_ids[train_end:val_end],
        "test": group_ids[val_end:],
    }
    split_dir = DATASET_DIR / "splits"
    split_dir.mkdir(exist_ok=True)
    for split_name, ids in partitions.items():
        paths = [str(path) for group_id in ids for path in groups[group_id]]
        (split_dir / f"{split_name}.txt").write_text("\n".join(paths) + "\n", encoding="utf-8")
    TRAIN_SOURCE = split_dir / "train.txt"
    VAL_SOURCE = split_dir / "val.txt"
    if TEST_SOURCE is None:
        TEST_SOURCE = split_dir / "test.txt"
    print("Không có validation: đã chia theo nhóm ảnh nguồn 80/10/10, seed=42.")

HAS_TEST = TEST_SOURCE is not None
config = {
    "path": str(DATASET_DIR),
    "train": str(TRAIN_SOURCE),
    "val": str(VAL_SOURCE),
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}
if HAS_TEST:
    config["test"] = str(TEST_SOURCE)
DATA_YAML = DATASET_DIR / "data_local.yaml"
with DATA_YAML.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, allow_unicode=True, sort_keys=False)

print("Các lớp:", CLASS_NAMES)
print("Số ảnh train:", len(get_images(TRAIN_SOURCE)))
print("Số ảnh validation:", len(get_images(VAL_SOURCE)))
print("Số ảnh test:", len(get_images(TEST_SOURCE)) if HAS_TEST else "Không có")
print("Cấu hình dùng để train:", DATA_YAML)


## 4b. Kiểm tra 6 lớp và tạo ánh xạ
Tên lớp phải khớp đúng 6 lệnh điều khiển. Dừng nếu thiếu hoặc dư lớp; ánh xạ theo **tên**, không theo thứ tự ID. `unknown` chỉ thuộc bộ phân loại ONNX cũ.

In [ ]:
import json

PROJECT_CLASSES = ["left", "no_left", "right", "no_right", "straight", "stop"]
actual_names = set(CLASS_NAMES.values())
missing = sorted(set(PROJECT_CLASSES) - actual_names)
extra = sorted(actual_names - set(PROJECT_CLASSES))
print("Các lớp còn thiếu:", missing or "Không thiếu")
print("Các lớp ngoài dự kiến:", extra or "Không có")
if missing or extra:
    raise ValueError(
        f"Dataset không khớp 6 lớp: thiếu={missing}, dư={extra}. "
        "Kiểm tra workspace/project/version; không tự đổi ID hoặc gán nhãn gần nghĩa."
    )

SOURCE_TO_PROJECT = {name: name for name in CLASS_NAMES.values()}
for class_id, source_name in CLASS_NAMES.items():
    print(f"YOLO ID {class_id}: {source_name} -> {SOURCE_TO_PROJECT[source_name]}")

MAPPING_PATH = DATASET_DIR / "sign_mapping.json"
MAPPING_PATH.write_text(json.dumps({
    "source_to_project": SOURCE_TO_PROJECT,
    "missing_project_classes": missing,
    "dataset": f"{DATASET_WORKSPACE}/{DATASET_PROJECT}",
    "version": DATASET_VERSION,
    "license": "CC BY 4.0",
    "source_url": "https://universe.roboflow.com/sa-tr/trafficsign-jryf1",
}, ensure_ascii=False, indent=2), encoding="utf-8")


## 5. Kiểm tra và xem nhãn
Nhãn YOLO: `class_id x_center y_center width height`, tọa độ chuẩn hóa trong `[0, 1]`.

Kiểm tra mẫu train/validation cho cả 6 lớp, sau đó xem ảnh có khung nhãn. Ảnh nền có thể không có file nhãn hoặc có file nhãn rỗng.

In [ ]:
train_images = get_images(TRAIN_SOURCE)
class_counts = {class_id: 0 for class_id in CLASS_NAMES}
missing_labels = 0
empty_labels = 0
labeled_images = []

def label_path(image_path):
    # Nhãn nằm cạnh thư mục images, kể cả khi đọc ảnh qua danh sách TXT.
    relative = image_path.relative_to(DATASET_DIR)
    label_parts = ["labels" if part == "images" else part for part in relative.parts]
    return (DATASET_DIR / Path(*label_parts)).with_suffix(".txt")

for image_path in train_images:
    annotation = label_path(image_path)
    if not annotation.exists():
        missing_labels += 1
        continue
    lines = annotation.read_text(encoding="utf-8").splitlines()
    lines = [line for line in lines if line.strip()]
    if not lines:
        empty_labels += 1
        continue
    labeled_images.append(image_path)
    for line_number, line in enumerate(lines, 1):
        values = line.split()
        assert len(values) == 5, f"Sai định dạng: {annotation}, dòng {line_number}"
        class_value, x, y, w, h = map(float, values)
        class_id = int(class_value)
        assert class_value == class_id and class_id in CLASS_NAMES, f"Sai ID lớp: {annotation}"
        assert all(0 <= v <= 1 for v in (x, y, w, h)) and w > 0 and h > 0, f"Sai tọa độ: {annotation}"
        class_counts[class_id] += 1

assert labeled_images, "Không có ảnh train nào chứa nhãn biển báo hợp lệ."
print("Ảnh thiếu file nhãn:", missing_labels)
print("Ảnh có file nhãn rỗng:", empty_labels)
print("Số bounding box theo lớp:")
for class_id, count in class_counts.items():
    print(f"  {CLASS_NAMES[class_id]}: {count}")

# Có tên trong YAML chưa đủ: mỗi lớp phải có bounding box để học.
missing_train = [CLASS_NAMES[i] for i, count in class_counts.items() if count == 0]
if missing_train:
    raise ValueError(f"Không có bounding box train cho các lớp: {missing_train}")

# Kiểm tra cả validation để mỗi biển đều có dữ liệu đánh giá.
val_counts = {i: 0 for i in CLASS_NAMES}
for image_path in get_images(VAL_SOURCE):
    annotation = label_path(image_path)
    if not annotation.exists():
        continue
    for line_number, line in enumerate(annotation.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        values = line.split()
        assert len(values) == 5, f"Sai nhãn validation: {annotation}, dòng {line_number}"
        class_value, x, y, w, h = map(float, values)
        class_id = int(class_value)
        assert class_value == class_id and class_id in CLASS_NAMES, f"Sai ID validation: {annotation}"
        assert all(0 <= v <= 1 for v in (x, y, w, h)) and w > 0 and h > 0, f"Sai tọa độ: {annotation}"
        val_counts[class_id] += 1
print("Số bounding box validation:", {CLASS_NAMES[i]: count for i, count in val_counts.items()})
missing_val = [CLASS_NAMES[i] for i, count in val_counts.items() if count == 0]
if missing_val:
    raise ValueError(f"Validation thiếu mẫu cho {missing_val}; cần kiểm tra/chia lại dữ liệu trước khi train.")

# Xem ít nhất một ảnh cho từng lớp, không chỉ lấy ngẫu nhiên bốn ảnh.
sample_images = []
for class_id in CLASS_NAMES:
    for image_path in labeled_images:
        rows = label_path(image_path).read_text(encoding="utf-8").splitlines()
        if any(line.strip() and int(float(line.split()[0])) == class_id for line in rows):
            if image_path not in sample_images:
                sample_images.append(image_path)
            break

for image_path in sample_images:
    with Image.open(image_path) as source:
        image = source.convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    for line in label_path(image_path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        class_id, x, y, w, h = map(float, line.split())
        box = ((x - w/2)*width, (y - h/2)*height, (x + w/2)*width, (y + h/2)*height)
        draw.rectangle(box, outline="red", width=3)
        draw.text((box[0], max(0, box[1] - 14)), str(int(class_id)), fill="red")
    print(image_path.name, "— số trên khung là ID lớp trong danh sách phía trên")
    image.thumbnail((900, 600))
    display(image)

## 6. Nạp mô hình
Fine-tune từ trọng số pretrained. Hết bộ nhớ GPU: giảm `BATCH_SIZE` xuống 4 hoặc 2 ở bước 2. Muốn nhẹ hơn: chọn `yolo11n.pt`.

Tắt lật ảnh để tránh làm sai ý nghĩa biển trái/phải; dùng `workers=0` để tương thích notebook trên Windows.

In [ ]:
model = YOLO(str(PRETRAINED_DIR / MODEL_NAME))
print(f"Model: {MODEL_NAME} | Epochs: {EPOCHS} | Image: {IMAGE_SIZE} | Batch: {BATCH_SIZE}")

## 7. Huấn luyện
Ultralytics tự in tiến độ từng epoch; không cần callback riêng. `best.pt` là checkpoint tốt nhất, `last.pt` dùng để tiếp tục lượt bị gián đoạn. Tạm tắt biểu đồ trong lúc train; validation, log và `results.csv` vẫn được lưu.

In [ ]:
print("Đang chuẩn bị dữ liệu và mô hình...", flush=True)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=0,
    patience=PATIENCE,
    optimizer="auto",
    seed=42,
    deterministic=True,
    fliplr=0.0,
    flipud=0.0,
    close_mosaic=10,
    project=str(RUNS_DIR),
    name="yolo11_bien_bao",
    exist_ok=False,
    save=True,
    plots=False,  # Có thể bật lại sau khi xác nhận môi trường chạy ổn định.
    verbose=True,
)

# Lấy đường dẫn thực tế, kể cả khi Ultralytics tự thêm số vào tên thư mục.
RUN_DIR = Path(model.trainer.save_dir).resolve()
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
assert BEST_PT.exists(), f"Chưa tìm thấy trọng số: {BEST_PT}"
print("Thư mục kết quả:", RUN_DIR)
print("Trọng số tốt nhất:", BEST_PT)

## 8. Biểu đồ huấn luyện
Xem loss, chỉ số validation và các lớp thường nhầm lẫn.

In [ ]:
for filename in ["results.png", "confusion_matrix_normalized.png"]:
    path = RUN_DIR / filename
    if path.exists():
        print(filename)
        display(Image.open(path))

## 9. Đánh giá validation
**Precision:** tỷ lệ dự đoán đúng. **Recall:** tỷ lệ biển thật được tìm thấy. **mAP50 / mAP50–95:** chất lượng detection tổng hợp; càng cao càng tốt.

In [ ]:
best_model = YOLO(str(BEST_PT))
val_metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=0,
    project=str(RUNS_DIR),
    name="danh_gia_validation",
    plots=True,
)
print(f"Precision: {val_metrics.box.mp:.4f}")
print(f"Recall:    {val_metrics.box.mr:.4f}")
print(f"mAP50:     {val_metrics.box.map50:.4f}")
print(f"mAP50-95:  {val_metrics.box.map:.4f}")

## 10. Đánh giá test
Chỉ dùng test để báo cáo sau khi chốt cấu hình bằng validation. Không có tập test thì bỏ qua.

In [ ]:
if HAS_TEST:
    test_metrics = best_model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=0,
        project=str(RUNS_DIR),
        name="danh_gia_test",
        plots=True,
    )
    print(f"Test mAP50:    {test_metrics.box.map50:.4f}")
    print(f"Test mAP50-95: {test_metrics.box.map:.4f}")
else:
    print("Dataset không có tập test. Chỉ mới có kết quả validation.")

## 11. Dự đoán thử
Ưu tiên ảnh test. `conf=0.25` là ngưỡng hiển thị dự đoán, có thể điều chỉnh.

In [ ]:
preview_folder = TEST_SOURCE if HAS_TEST else VAL_SOURCE
preview_images = get_images(preview_folder)[:4]
predictions = best_model.predict(
    source=[str(p) for p in preview_images],
    imgsz=IMAGE_SIZE,
    conf=0.25,
    device=DEVICE,
    save=True,
    project=str(RUNS_DIR),
    name="du_doan_mau",
)
for result in predictions:
    # plot() trả ảnh BGR; đổi sang RGB để matplotlib hiển thị đúng màu.
    plt.figure(figsize=(12, 7))
    plt.imshow(result.plot()[:, :, ::-1])
    plt.axis("off")
    plt.title(Path(result.path).name)
    plt.show()

## 12. Chuyển sang ONNX
Xuất từ **best.pt**, đầu vào cố định `1 × 3 × IMAGE_SIZE × IMAGE_SIZE`, FP32, opset 12. Sau đó kiểm tra cấu trúc và chạy thử một ảnh validation trên CPU.

[Tài liệu ONNX của Ultralytics](https://docs.ultralytics.com/integrations/onnx/).

In [ ]:
import onnx

# Nạp lại checkpoint tốt nhất, tránh xuất nhầm trọng số epoch cuối.
export_model = YOLO(str(BEST_PT))
ONNX_PATH = Path(export_model.export(
    format="onnx",
    imgsz=IMAGE_SIZE,
    batch=1,
    dynamic=False,
    simplify=False,
    opset=12,
    device="cpu",
)).resolve()

onnx.checker.check_model(str(ONNX_PATH))
onnx_model = YOLO(str(ONNX_PATH), task="detect")
sample_image = get_images(VAL_SOURCE)[0]
onnx_results = onnx_model.predict(
    source=str(sample_image),
    imgsz=IMAGE_SIZE,
    conf=0.25,
    device="cpu",
    verbose=False,
)
print("ONNX hợp lệ, đã chạy thử:", ONNX_PATH)
print(f"Dung lượng: {ONNX_PATH.stat().st_size / (1024 ** 2):.1f} MB")
print("Số biển dự đoán trên ảnh thử:", len(onnx_results[0].boxes))


## 13. Lưu kết quả
Lưu cả `.pt`, `.onnx`, ánh xạ lớp và ZIP kết quả. Trên Colab, tải ZIP về trước khi phiên chạy kết thúc. API key không nằm trong file xuất.

In [ ]:
EXPORT_DIR = PROJECT_ROOT / "models" / "yolo11"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = EXPORT_DIR / "bien_bao_best.pt"
shutil.copy2(BEST_PT, MODEL_PATH)
ONNX_MODEL_PATH = EXPORT_DIR / "bien_bao_best.onnx"
shutil.copy2(ONNX_PATH, ONNX_MODEL_PATH)
shutil.copy2(DATA_YAML, EXPORT_DIR / DATA_YAML.name)

versions = {
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "roboflow": importlib.metadata.version("roboflow"),
    "onnx": importlib.metadata.version("onnx"),
    "onnxruntime": importlib.metadata.version("onnxruntime"),
}
(EXPORT_DIR / "versions.txt").write_text(
    "\n".join(f"{name}=={value}" for name, value in versions.items()), encoding="utf-8"
)
shutil.copy2(EXPORT_DIR / "versions.txt", RUN_DIR / "versions.txt")
shutil.copy2(DATA_YAML, RUN_DIR / DATA_YAML.name)
# Đặt sidecar cạnh cả hai bản trọng số; ZIP chứa ánh xạ cạnh best.pt.
shutil.copy2(MAPPING_PATH, EXPORT_DIR / "sign_mapping.json")
shutil.copy2(MAPPING_PATH, RUN_DIR / "weights" / "sign_mapping.json")
ZIP_PATH = shutil.make_archive(str(WORK_DIR / f"{RUN_DIR.name}_ket_qua"), "zip", root_dir=RUN_DIR)
print("PyTorch:", MODEL_PATH)
print("ONNX:", ONNX_MODEL_PATH)
print("Kết quả đóng gói:", ZIP_PATH)

# Trên Google Colab, bỏ dấu # ở hai dòng dưới để tải ZIP về máy.
# from google.colab import files
# files.download(ZIP_PATH)


## 14. Ảnh/video riêng và tiếp tục train
Bỏ dấu `#`, thay đường dẫn để chạy. Resume chỉ dùng cho lượt train bị gián đoạn.

In [ ]:
# Dự đoán ảnh riêng:
# detector = YOLO(str(MODEL_PATH))
# detector.predict(source="duong_dan/anh.jpg", conf=0.25, save=True)

# Dự đoán video riêng:
# detector = YOLO(str(MODEL_PATH))
# for result in detector.predict(source="duong_dan/video.mp4", conf=0.25, save=True, stream=True):
#     pass  # stream=True giúp không giữ kết quả của cả video trong RAM.

# Tiếp tục một lượt train bị gián đoạn:
# resumed_model = YOLO("outputs/runs/yolo11_bien_bao/weights/last.pt")
# resumed_model.train(resume=True, workers=0)

## Lưu ý
Kiểm tra ảnh gần trùng giữa các tập và đánh giá mô hình trên camera VIA thực tế. Đủ 6 tên lớp không bảo đảm độ chính xác trong mô phỏng.

Tài liệu: [YOLO11](https://docs.ultralytics.com/models/yolo11/) · [Train](https://docs.ultralytics.com/modes/train/) · [Roboflow dataset](https://universe.roboflow.com/sa-tr/trafficsign-jryf1).

## 15. Tích hợp vào chương trình hiện tại
Copy **cả** `bien_bao_best.pt` và `sign_mapping.json` từ thư mục `models/yolo11` về cùng một thư mục trên máy chạy xe. Nếu dùng file ZIP, hai file `weights/best.pt` và `weights/sign_mapping.json` cũng đã nằm cạnh nhau.

Trong PowerShell ở thư mục gốc repository:
```powershell
python -m pip install -r requirements.txt
$env:TRAFFIC_SIGN_MODEL = "D:\duong_dan_model\bien_bao_best.pt"
python -m simulator.drive
```

Bộ đọc mới chạy YOLO trên toàn ảnh, dùng `model.names` và `sign_mapping.json` đổi sang tên điều khiển. Những lớp không có ánh xạ được bỏ qua; không thay đổi thuật toán bám làn. Dataset mới có tên lớp khớp đủ sáu lệnh. Sau khi train, cần kiểm tra độ chính xác từng lớp và tốc độ thực tế trong mô phỏng trước khi điều chỉnh tham số điều khiển.

Nguồn đối chiếu lớp: https://universe.roboflow.com/sa-tr/trafficsign-jryf1

File YOLO ONNX là mô hình detection toàn ảnh, không thay trực tiếp cho bộ phân loại LeNet ONNX 32×32 cũ. Dùng `YOLO("bien_bao_best.onnx", task="detect")` để dự đoán riêng; phần tích hợp xe ở trên vẫn dùng `.pt`.
